# `object`, `type` and `type()`: clearing up the confusion

People mix up three separate things:

| Name | What it is | One-line definition |
|------|-----------|---------------------|
| `object` | a **class** | The **root of the inheritance tree**. Every class inherits from it. |
| `type` | a **class** (a *metaclass*) | The class whose instances are classes. Every class is an **instance** of it. |
| `type(x)` | a **call** of that class with 1 argument | Returns the class of `x`. |
| `type(name, bases, ns)` | a **call** of that class with 3 arguments | Creates a new class. |

There is no separate "type function". `type` is one class. Calling it with 1 argument asks
*"what is your class?"*. Calling it with 3 arguments says *"build me a class"*.

### Two relationships, kept separate
Most of the confusion comes from mixing up two different relationships:

1. **is-instance-of**: what built this object? → `type(x)`, `x.__class__`, `isinstance(x, C)`
2. **is-subclass-of**: what does this class inherit from? → `C.__bases__`, `C.__mro__`, `issubclass(C, B)`

```
         instance-of ──►              subclass-of ──►

   42  ──────────►  int  ──────────►  object
                     │                  ▲
                     │ instance-of      │ subclass-of
                     ▼                  │
                   type  ───────────────┘
                     │ ▲
                     └─┘  instance-of (type is an instance of itself)
```

Keep this picture in mind. Every section below fills in part of it.

## 1. Everything is an object, and every object has a type

In Python, *everything* is an object: numbers, strings, functions, modules, classes, even `None`.
Every object has three things:

- **identity**: `id(x)`, which never changes during its lifetime
- **type**: `type(x)`, which decides which operations it supports
- **value**: its data

In [ ]:
things = [42, 3.14, "hi", [1, 2], None, len, print, int, type]

for t in things:
    print(f"{t!r:<32} type -> {type(t)}")

In [ ]:
def greet():
    return "hello"

import math

print(type(greet))   # functions are objects of class 'function'
print(type(math))    # modules are objects of class 'module'
print(type(int))     # classes are objects too, of class 'type'
print(isinstance(greet, object), isinstance(math, object), isinstance(int, object))

> **Key idea:** `isinstance(anything, object)` is **always** `True`. That is what
> "everything is an object" means in practice.

## 2. `object`: the base class of everything

`object` is the root of **every** inheritance chain. In Python 3 these three are identical:

```python
class A: ...
class A(): ...
class A(object): ...   # Python 2 habit, redundant in Python 3
```

In [ ]:
class A:
    pass

print(A.__bases__)    # (<class 'object'>,)  -- added for you
print(A.__mro__)      # A first, object last
print(int.__mro__, bool.__mro__, sep="\n")
print(object.__bases__)  # () -- object is the top, it has no parent

### What does `object` give you for free?

Every class inherits a minimal set of **default behaviours** from `object`. That is why a
completely empty class can still be printed, compared, hashed and so on.

In [ ]:
print(dir(object))

In [ ]:
class Empty:
    pass

e1, e2 = Empty(), Empty()

print(repr(e1))          # object.__repr__  -> <__main__.Empty object at 0x...>
print(e1 == e2)          # object.__eq__    -> identity comparison: False
print(e1 == e1)          # True, same object
print(hash(e1) == hash(e1))  # object.__hash__ -> based on id
print(e1.__class__)      # object provides __class__
print(e1.__dict__)       # instance attribute storage (added by normal classes)

The important defaults to know for interviews:

| Inherited from `object` | Default behaviour | You usually override it to... |
|------------------------|-------------------|------------------------------|
| `__new__` | allocates the instance | control creation (singletons, immutables) |
| `__init__` | does nothing | set up attributes |
| `__repr__` / `__str__` | `<Cls object at 0x..>`; `__str__` falls back to `__repr__` | give readable output |
| `__eq__` | identity (`is`) | compare by value |
| `__hash__` | based on `id()` | keep consistent with `__eq__` |
| `__getattribute__`, `__setattr__`, `__delattr__` | normal attribute lookup | intercept attribute access |
| `__init_subclass__` | does nothing | hook into subclass creation (section 8) |

> **Gotcha:** if you define `__eq__` but not `__hash__`, Python sets `__hash__ = None` and
> instances become **unhashable**, so they can no longer go in sets or be dict keys.

In [ ]:
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __eq__(self, other):
        return isinstance(other, Point) and (self.x, self.y) == (other.x, other.y)

print(Point(1, 2) == Point(1, 2))   # True now
print(Point.__hash__)               # None! defining __eq__ removed the inherited hash
try:
    {Point(1, 2)}
except TypeError as err:
    print("TypeError:", err)

### A bare `object()` instance

You can instantiate `object` directly. The result is about as minimal as an object gets: it has
**no `__dict__`**, so you can't add attributes to it. It is also unique, so it only equals itself.

In [ ]:
o = object()
print(o, o == object())
try:
    o.name = "x"
except AttributeError as err:
    print("AttributeError:", err)

**Real-world use: sentinels.** Sometimes `None` is a valid value, so it can't also mean
"argument not passed". A unique `object()` solves this. The standard library does the same:
`dataclasses.MISSING` and the private `_sentinel` objects in `functools` work this way.

In [ ]:
_MISSING = object()

def get_setting(settings, key, default=_MISSING):
    if key in settings:
        return settings[key]
    if default is _MISSING:            # the caller passed nothing at all
        raise KeyError(f"required setting {key!r} is missing")
    return default                      # the caller passed a default, possibly None

cfg = {"debug": None}
print(get_setting(cfg, "debug"))            # None is a real value here
print(get_setting(cfg, "timeout", None))    # explicit None default works
try:
    get_setting(cfg, "timeout")
except KeyError as err:
    print("KeyError:", err)

## 3. `type`: the class of classes (a metaclass)

A class is an object too. If every object has a type, what is the type of a class?

**`type`.** Classes are *instances of* `type`. A class that produces classes is called a
**metaclass**, and `type` is the default metaclass.

In [ ]:
class Dog:
    pass

rex = Dog()

print(type(rex))      # <class 'Dog'>    rex  is an instance of Dog
print(type(Dog))      # <class 'type'>   Dog  is an instance of type
print(type(int), type(str), type(object))   # all built-in classes too
print(type(type))     # <class 'type'>   type is an instance of ITSELF

### The `object` ↔ `type` loop

This is the part people find hardest. Read each line as a separate fact:

In [ ]:
# instance-of relationships
print(isinstance(object, type))   # True  -> object is a class, and every class is an instance of type
print(isinstance(type, type))     # True  -> type is an instance of itself
print(isinstance(type, object))   # True  -> everything is an object

# subclass-of relationships
print(issubclass(type, object))   # True  -> type inherits from object (like every class)
print(issubclass(object, type))   # False -> object does NOT inherit from type
print(type.__bases__)             # (<class 'object'>,)

**How can `type` be an instance of itself, and `object` be an instance of `type`, while `type`
is a subclass of `object`?** It's circular, and you can't build that loop in pure Python. CPython
wires up these two objects by hand in C when the interpreter starts. Every other class sits
inside this structure:

```
                    ┌──────── instance-of ────────┐
                    ▼                             │
   object ◄── subclass-of ── type ────────────────┘
      ▲                       │  (type's class is type)
      │ instance-of           │
      └───────────────────────┘  (object's class is type)

   Dog ── subclass-of ──► object         type(Dog) is type
   rex ── instance-of ──► Dog            type(rex) is Dog
```

**One sentence to remember:** *`object` is at the top of the **inheritance** chain;
`type` is at the top of the **instance-of** chain.*

### Classes have attributes because they are objects

Since a class is an ordinary object (an instance of `type`), you can pass it around, store it in a
list, give it attributes and return it from functions. Frameworks rely on this all the time.

In [ ]:
class Cat:
    sound = "meow"

animal_classes = [Dog, Cat]          # classes in a list
Dog.sound = "woof"                   # add an attribute to a class object at runtime

def make(cls):                       # pass a class as an argument
    return cls()

for cls in animal_classes:
    print(cls.__name__, cls.sound, make(cls))

## 4. `type(x)` with one argument: "what is your class?"

`type(x)` returns the exact class that created `x`. It is the same as `x.__class__`.

In [ ]:
print(type(3.0) is float, (3.0).__class__ is float)
print(type([]) is list)
print(type(True))          # bool, the exact class, not int

### `type(x) ==` vs `isinstance(x, ...)`: an interview favourite

- `type(x) is C` checks the **exact** class and **ignores inheritance**.
- `isinstance(x, C)` returns `True` for `C` **and any subclass of `C`**, so it respects
  polymorphism and the Liskov substitution principle.

Use `isinstance` by default. Use an exact `type()` check only when subclasses really must be treated differently.

In [ ]:
class Animal: ...
class Bird(Animal): ...

tweety = Bird()
print(type(tweety) is Animal)       # False: the exact class is Bird
print(isinstance(tweety, Animal))   # True: a Bird IS-A Animal

# Classic surprise: bool is a subclass of int
print(isinstance(True, int))        # True
print(type(True) is int)            # False

**Where an exact type check is right.** JSON-like validation, where `True` must **not** pass as an
integer. This bug shows up in real APIs: a `quantity=True` order sneaks through `isinstance(v, int)`.

In [ ]:
def validate_quantity(value):
    # isinstance(value, int) would wrongly accept True/False
    if type(value) is not int:
        raise TypeError(f"quantity must be int, got {type(value).__name__}")
    return value

print(validate_quantity(3))
for bad in (True, 3.0, "3"):
    try:
        validate_quantity(bad)
    except TypeError as err:
        print("TypeError:", err)

Tips:
- Compare with `is`, not `==`: `type(x) is int`. Classes are singletons, and `is` is clearer.
- `isinstance` accepts a tuple, or a union in Python 3.10+: `isinstance(x, (int, float))` or `isinstance(x, int | float)`.
- Prefer **duck typing** when you can: don't check the type at all, just call the method.

### `type(self)` inside methods: subclass-friendly code

Writing the class name directly (`Money(...)`) inside a method breaks for subclasses: they get
an instance of the parent class back. `type(self)(...)` builds an instance of
*whatever class `self` actually is*. `@classmethod` alternative constructors use the same idea
through `cls`.

In [ ]:
class Money:
    def __init__(self, amount):
        self.amount = amount

    def __add__(self, other):
        return type(self)(self.amount + other.amount)    # not Money(...)

    def __repr__(self):
        return f"{type(self).__name__}({self.amount})"    # not hard-coded "Money"

    @classmethod
    def zero(cls):
        return cls(0)

class Euro(Money):
    pass

print(Euro(5) + Euro(7))   # Euro(12), stays a Euro
print(Euro.zero())         # Euro(0)

## 5. `type(name, bases, namespace)` with three arguments: "build me a class"

The `class` statement is mostly **syntactic sugar** for calling `type`. These two are equivalent:

In [ ]:
# 1) the usual way
class User:
    role = "guest"

    def greet(self):
        return f"Hi, my role is {self.role}"


# 2) the same class, built by calling type(name, bases, namespace)
def greet(self):
    return f"Hi, my role is {self.role}"

User2 = type(
    "User2",              # name     -> becomes __name__
    (object,),            # bases    -> tuple of parent classes ((object,) can also be ())
    {"role": "guest",     # namespace -> class body: attributes and methods
     "greet": greet},
)

print(User().greet(), "|", User2().greet())
print(type(User), type(User2))
print(User2.__name__, User2.__bases__)

In [ ]:
# Inheritance works too: pass bases
Admin = type("Admin", (User,), {"role": "admin"})
print(Admin().greet(), Admin.__mro__)

### What actually happens when Python runs a `class` statement

1. Python works out the **metaclass**: `type` by default, or `metaclass=...`, or the parent's metaclass.
2. It calls `metaclass.__prepare__(name, bases)` to get an empty **namespace** (a dict).
3. It **executes the class body** like a function body, inside that namespace. Every `def` and every
   assignment becomes a key in it.
4. It calls **`metaclass(name, bases, namespace)`**, which is `type(name, bases, ns)` by default.
   That call creates the class object.
5. It binds the result to the class name in the enclosing scope.

So `class` = "run the body to fill a dict, then call `type(...)`". Once you see this, metaclasses
stop seeming mysterious.

### Real-world use: generating classes at runtime

You rarely need `type()` with 3 arguments in application code. It is handy when the **class
shape is only known at runtime**, from config, a schema or a database.

**Example A: a family of exception classes generated from error codes** (common in API client libraries):

In [ ]:
class APIError(Exception):
    status = None

ERROR_CODES = {400: "BadRequest", 401: "Unauthorized", 404: "NotFound", 429: "RateLimited"}

# build one subclass per status code
ERRORS = {
    status: type(name + "Error", (APIError,), {"status": status})
    for status, name in ERROR_CODES.items()
}

def raise_for_status(status):
    if status in ERRORS:
        raise ERRORS[status](f"HTTP {status}")

try:
    raise_for_status(404)
except APIError as err:       # caught by the common base class
    print(type(err).__name__, err.status, err)

**Example B: building model classes from a schema** (simplified version of what ORMs, Pydantic's
`create_model`, and `collections.namedtuple` / `dataclasses.make_dataclass` do internally):

In [ ]:
def make_model(name, fields):
    def __init__(self, **kwargs):
        for field, field_type in fields.items():
            value = kwargs[field]
            if not isinstance(value, field_type):
                raise TypeError(f"{field} must be {field_type.__name__}")
            setattr(self, field, value)

    def __repr__(self):
        args = ", ".join(f"{f}={getattr(self, f)!r}" for f in fields)
        return f"{name}({args})"

    return type(name, (), {"__init__": __init__, "__repr__": __repr__, "fields": fields})

schema = {"id": int, "email": str}          # imagine this came from JSON / a DB table
Customer = make_model("Customer", schema)
print(Customer(id=1, email="a@b.com"))

# The standard library does the same for you:
from dataclasses import make_dataclass
Product = make_dataclass("Product", [("sku", str), ("price", float)])
print(Product("X1", 9.99), type(Product))

## 6. Metaclasses: subclassing `type`

`type` is a class, so you can **subclass it**. A subclass of `type` is a custom metaclass. It
controls **how classes are created**, just as an ordinary class controls how its instances are created.

| Level | Created by | Customise with |
|-------|-----------|----------------|
| instance `rex` | class `Dog` | `Dog.__new__` / `Dog.__init__` |
| class `Dog` | metaclass `type` | `Meta.__new__` / `Meta.__init__` |

> Tim Peters: *"Metaclasses are deeper magic than 99% of users should ever worry about."*
> Know how they work, but reach for simpler tools first (section 8).

In [ ]:
class LoggingMeta(type):
    def __new__(mcls, name, bases, namespace):
        print(f"creating class {name!r} with attrs {[k for k in namespace if not k.startswith('__')]}")
        return super().__new__(mcls, name, bases, namespace)

class Service(metaclass=LoggingMeta):   # the message prints at class DEFINITION time
    def run(self): ...

print(type(Service))                     # LoggingMeta, not type
print(isinstance(Service, type))         # still a type, since LoggingMeta subclasses type

### Real-world use: a singleton via a metaclass

`Cls()` is really a call to `type(Cls).__call__(Cls)`. The `__call__` on `type` runs `__new__`
and then `__init__`. Override `__call__` in a metaclass and you control instantiation itself.

In [ ]:
class SingletonMeta(type):
    _instances = {}

    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)   # runs __new__ + __init__
        return cls._instances[cls]

class Config(metaclass=SingletonMeta):
    def __init__(self):
        print("loading config...")    # runs only once

a, b = Config(), Config()
print(a is b)

### Real-world use: enforcing rules on classes

Frameworks use metaclasses to check or rewrite a class when it is *defined*. Django's
`ModelBase` collects `Field`s into `_meta`. `enum.EnumType` (formerly `EnumMeta`) turns attributes into members.
`abc.ABCMeta` tracks abstract methods. Here is a small version:

In [ ]:
class InterfaceMeta(type):
    required = ("execute",)

    def __init__(cls, name, bases, namespace):
        super().__init__(name, bases, namespace)
        if bases:   # skip the abstract root class itself
            missing = [m for m in cls.required if m not in namespace]
            if missing:
                raise TypeError(f"{name} must define {missing}")

class Command(metaclass=InterfaceMeta):
    pass

class Deploy(Command):
    def execute(self):
        return "deploying"

try:
    class Broken(Command):     # fails right away, at definition time
        pass
except TypeError as err:
    print("TypeError:", err)

import abc, enum
print(type(abc.ABC), type(enum.Enum))   # standard library metaclasses

## 7. Putting it together: the full lookup picture

| Expression | Answer | Why |
|-----------|--------|-----|
| `type(rex)` | `Dog` | rex was created by `Dog` |
| `type(Dog)` | `type` | Dog was created by `type` |
| `type(type)` | `type` | bootstrapped loop |
| `type(object)` | `type` | object is a class, and classes are made by `type` |
| `Dog.__bases__` | `(object,)` | implicit parent |
| `type.__bases__` | `(object,)` | type is a class like any other |
| `object.__bases__` | `()` | top of the inheritance tree |
| `isinstance(x, object)` | always `True` | everything is an object |
| `isinstance(C, type)` | `True` for every class | every class is an instance of a metaclass derived from `type` |

In [ ]:
checks = {
    "type(rex) is Dog": type(rex) is Dog,
    "type(Dog) is type": type(Dog) is type,
    "type(type) is type": type(type) is type,
    "type(object) is type": type(object) is type,
    "Dog.__bases__ == (object,)": Dog.__bases__ == (object,),
    "type.__bases__ == (object,)": type.__bases__ == (object,),
    "object.__bases__ == ()": object.__bases__ == (),
    "isinstance(len, object)": isinstance(len, object),
    "isinstance(Config, type)": isinstance(Config, type),
}
for expr, result in checks.items():
    print(f"{expr:<30} {result}")
assert all(checks.values())

## 8. Real-world toolbox: what to use in practice

Pick the simplest tool that does the job:

| Need | Use |
|------|-----|
| check if a value is "some kind of X" | `isinstance(x, X)` |
| check the exact class, ignoring subclasses | `type(x) is X` |
| create an instance of the same class as `self` | `type(self)(...)` or `cls(...)` in a classmethod |
| print the class name | `type(self).__name__` |
| a unique "not provided" marker | `_MISSING = object()` |
| react when someone subclasses your class (registries, validation) | **`__init_subclass__`** |
| generate a class at runtime | `type(name, bases, ns)`, `make_dataclass`, `namedtuple` |
| change *how classes are built or called* | metaclass (last resort) |
| annotate "a class, not an instance" | `type[X]` |

### `__init_subclass__`: the metaclass alternative you should reach for first

Python 3.6 added `object.__init_subclass__`. It is called on the parent class whenever a subclass is
defined. It handles most of what people used to write metaclasses for, such as plugin registries,
and it avoids metaclass conflicts.

In [ ]:
class Exporter:
    registry = {}

    def __init_subclass__(cls, fmt, **kwargs):
        super().__init_subclass__(**kwargs)
        Exporter.registry[fmt] = cls          # auto-register every subclass

    def export(self, data): raise NotImplementedError

class CSVExporter(Exporter, fmt="csv"):
    def export(self, data):
        return ",".join(map(str, data))

class JSONExporter(Exporter, fmt="json"):
    def export(self, data):
        import json
        return json.dumps(data)

def export(data, fmt):
    return Exporter.registry[fmt]().export(data)   # no if/elif chain over types

print(Exporter.registry)
print(export([1, 2, 3], "csv"), export([1, 2, 3], "json"))

### Dispatching on type without `if type(x) == ...` chains

Long `if type(x) is A: ... elif type(x) is B: ...` chains are a code smell. Use polymorphism, or
`functools.singledispatch` when you can't add methods to the classes (for example built-ins):

In [ ]:
from functools import singledispatch
from datetime import date
from decimal import Decimal

@singledispatch
def to_json(value):
    raise TypeError(f"can't serialize {type(value).__name__}")

@to_json.register
def _(value: date):
    return value.isoformat()

@to_json.register
def _(value: Decimal):
    return str(value)

print(to_json(date(2026, 11, 6)), to_json(Decimal("9.99")))
try:
    to_json(object())
except TypeError as err:
    print("TypeError:", err)

### Type hints: `type[X]` means "the class itself"

In [ ]:
def build(cls: type[Exporter]) -> Exporter:   # takes a class, returns an instance
    return cls()

print(build(CSVExporter).export(["a", "b"]))

## 9. Common pitfalls

1. **`type(x) == SomeClass` to check a "kind"** breaks subclassing → use `isinstance`.
2. **Forgetting `bool` is an `int`** → `isinstance(True, int)` is `True`.
3. **Defining `__eq__` without `__hash__`** → instances become unhashable.
4. **Thinking `object` is a metaclass, or `type` is just a function** → `object` is the root *base
   class*; `type` is the root *metaclass*. Both are classes.
5. **Using a metaclass when `__init_subclass__`, a class decorator or a plain function would do**
   → metaclasses don't combine well: two bases with unrelated metaclasses raise a `TypeError`.
6. **Hard-coding the class name** in `__repr__`, `__add__` or factory methods → use `type(self)` / `cls`.
7. **Using `None` as the "not passed" default** when `None` is a valid value → use an `object()` sentinel.

In [ ]:
# Pitfall 5 demo: metaclass conflict
class MetaA(type): ...
class MetaB(type): ...
class X(metaclass=MetaA): ...
class Y(metaclass=MetaB): ...

try:
    class Z(X, Y): ...
except TypeError as err:
    print("TypeError:", err)

## 10. Interview cheat sheet: short answers you can give out loud

**Q: What is `object` in Python?**
The root base class. Every class inherits from it, implicitly in Python 3. It supplies the default
dunder methods: `__init__`, `__new__`, `__repr__`, `__eq__` (by identity), `__hash__` (by id),
attribute access and `__init_subclass__`.

**Q: What is `type`?**
The built-in metaclass, meaning the class whose instances are classes. `type(int)`, `type(MyClass)` and
even `type(type)` all return `type`.

**Q: Is `type()` a function?**
No. It is a class, and calling it has two forms. With one argument it returns the object's class (same as
`x.__class__`). With three arguments `(name, bases, namespace)` it creates a new class. The `class`
statement does the three-argument form for you.

**Q: How do `object` and `type` relate?**
`type` is a subclass of `object`, and `object` is an instance of `type`. `type` is also an instance of
itself. The interpreter sets up this loop in C at startup. `object` is the top of the inheritance
chain; `type` is the top of the instance-of chain.

**Q: `type(x) is C` vs `isinstance(x, C)`?**
`type` checks the exact class. `isinstance` also accepts subclasses, so it respects polymorphism and LSP.
Default to `isinstance`. Use exact checks only when subclasses must be rejected, for example
`bool` vs `int` in validation.

**Q: What is a metaclass and when would you use one?**
A subclass of `type` that customises class creation (`__new__` / `__init__`) or instantiation
(`__call__`). Examples: Django models, `Enum`, `ABCMeta`, singletons. In modern Python prefer
`__init_subclass__` or class decorators, and keep metaclasses for framework-level code.

**Q: What happens when Python executes `class Foo(Base): ...`?**
It picks the metaclass, prepares a namespace, runs the body inside that namespace, then calls
`metaclass("Foo", (Base,), namespace)` and binds the result to `Foo`.

**Q: What does `Foo()` actually do?**
It calls `type(Foo).__call__(Foo)`, which runs `Foo.__new__(Foo)` to create the instance and then
`__init__` on it, as long as `__new__` returned an instance of `Foo`.

In [ ]:
# Proof for the last answer: Foo() goes through type.__call__ -> __new__ -> __init__
class Foo:
    def __new__(cls, *args):
        print("1. __new__ creates the instance")
        return super().__new__(cls)

    def __init__(self, value):
        print("2. __init__ initializes it")
        self.value = value

f1 = Foo(10)
f2 = type.__call__(Foo, 20)    # exactly what Foo(20) does
print(f1.value, f2.value)

## 11. Practice exercises

Try each one before you open the solution cell below it.

**Exercise 1.** Without running code, predict the output, then check:
```python
print(type(type(42)), type(object) is type(int), issubclass(bool, object), isinstance(int, object))
```

In [ ]:
# Solution 1
print(type(type(42)), type(object) is type(int), issubclass(bool, object), isinstance(int, object))
# type(42) -> int; type(int) -> type.  type(object) and type(int) are both type -> True.
# bool -> int -> object, so True.  int is an object -> True.

**Exercise 2.** Create a class `Vector` with `type()` only, no `class` keyword. It needs an
`__init__(self, x, y)` and an `__abs__` that returns the length. Then make `Vector3D` inherit from it,
again using `type()`.

In [ ]:
# Solution 2
import math

Vector = type("Vector", (), {
    "__init__": lambda self, x, y: setattr(self, "xy", (x, y)),
    "__abs__": lambda self: math.hypot(*self.xy),
})
Vector3D = type("Vector3D", (Vector,), {})

print(abs(Vector(3, 4)), issubclass(Vector3D, Vector), type(Vector3D))

**Exercise 3.** Write a base class `Handler` that uses `__init_subclass__` to require every subclass
to set a class attribute `event: str`, and records it in `Handler.handlers`. Defining a subclass
without `event` should raise `TypeError`.

In [ ]:
# Solution 3
class Handler:
    handlers = {}

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        event = cls.__dict__.get("event")
        if not isinstance(event, str):
            raise TypeError(f"{cls.__name__} must define class attribute 'event: str'")
        Handler.handlers[event] = cls

class OnSignup(Handler):
    event = "user.signup"

try:
    class NoEvent(Handler):
        pass
except TypeError as err:
    print("TypeError:", err)

print(Handler.handlers)

**Exercise 4.** Fix this class so subclasses get the correct class back from `scaled` and
`__repr__`, and so instances stay usable in a `set`:
```python
class Temperature:
    def __init__(self, deg): self.deg = deg
    def scaled(self, k): return Temperature(self.deg * k)
    def __repr__(self): return f"Temperature({self.deg})"
    def __eq__(self, other): return self.deg == other.deg
```

In [ ]:
# Solution 4
class Temperature:
    def __init__(self, deg):
        self.deg = deg

    def scaled(self, k):
        return type(self)(self.deg * k)

    def __repr__(self):
        return f"{type(self).__name__}({self.deg})"

    def __eq__(self, other):
        if not isinstance(other, Temperature):
            return NotImplemented
        return self.deg == other.deg

    def __hash__(self):                  # restore hashability, consistent with __eq__
        return hash(self.deg)

class Celsius(Temperature):
    pass

print(Celsius(10).scaled(2), {Celsius(1), Celsius(1)})

## Summary

- **Everything is an object.** `isinstance(anything, object)` is `True`.
- **`object`** is the root **base class**. It provides default behaviour; `class A:` means `class A(object):`.
- **`type`** is the root **metaclass**. Classes are instances of it, including `type` itself.
- **`type(x)`** gives the exact class of `x`. Prefer **`isinstance`** for checks, and use **`type(self)`** to stay subclass-friendly.
- **`type(name, bases, ns)`** builds a class. It is what the `class` statement does underneath.
- **Metaclasses** subclass `type` to customise class creation. Try **`__init_subclass__`** first.